# 01 Dataset Setup

This notebook prepares a reproducible Google Colab workflow for loading the MIMIC-III zip archive from Google Drive, extracting only once, validating the required tables, and writing a setup manifest for downstream notebooks.

Outputs from this notebook are written to `results/manifests/` and the extracted CSV tables live under the configured data directory.

## Why this notebook exists

- Keep dataset access reproducible across Colab sessions.
- Avoid hidden notebook state by saving paths and validation outputs.
- Support large MIMIC CSV files with selective loading utilities.
- Prepare a foundation for cohort construction without data leakage shortcuts.

In [ ]:
# Colab bootstrap
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/multimodal-early-sepsis') if IN_COLAB else Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
# Optional: install missing dependencies in Colab
if IN_COLAB:
    %pip -q install pyyaml pandas

In [ ]:
from src.utils.config import load_config
from src.utils.paths import ensure_directories, resolve_project_paths
from src.utils.logging_utils import write_run_manifest
from src.data_processing.dataset_setup import find_zip_candidates, unzip_dataset
from src.data_processing.io import list_available_tables, validate_required_tables

In [ ]:
base_config_path = PROJECT_ROOT / 'configs' / 'base.yaml'
override_config_path = PROJECT_ROOT / 'configs' / 'colab.yaml' if IN_COLAB else None
config = load_config(base_config_path, override_config_path)
paths = resolve_project_paths(config)
ensure_directories(paths)
config

## Locate the zip archive

The default search root is your Google Drive root in Colab. If there are multiple zip files, pick the correct one explicitly.

In [ ]:
search_root = Path(config['colab']['default_drive_search_root']) if IN_COLAB else paths['project_root']
zip_candidates = find_zip_candidates(
    search_root=search_root,
    filename_patterns=config['dataset']['zip_filename_patterns'],
)
zip_candidates[:20]

In [ ]:
# Set this manually if more than one candidate appears.
ZIP_PATH = zip_candidates[0] if zip_candidates else None
ZIP_PATH

In [ ]:
assert ZIP_PATH is not None, 'No MIMIC zip archive found. Set ZIP_PATH manually.'
extracted_members = unzip_dataset(
    zip_path=ZIP_PATH,
    destination_dir=paths['extracted_data_dir'],
    overwrite=False,
)
len(extracted_members)

In [ ]:
available_tables = list_available_tables(paths['extracted_data_dir'])
required_status = validate_required_tables(
    paths['extracted_data_dir'],
    config['dataset']['required_tables'],
)
available_tables[:20], required_status

In [ ]:
manifest_path = paths['manifests_dir'] / '01_dataset_setup_manifest.json'
write_run_manifest(
    path=manifest_path,
    stage='01_dataset_setup',
    config=config,
    extra={
        'zip_path': str(ZIP_PATH),
        'search_root': str(search_root),
        'extracted_dir': str(paths['extracted_data_dir']),
        'available_tables': available_tables,
        'required_status': required_status,
    },
)
manifest_path

## Sanity-check one table with selective loading

This final check shows how later notebooks will load only the necessary columns instead of reading entire MIMIC tables into memory.

In [ ]:
from src.data_processing.io import load_table

patients_preview = load_table(
    extracted_dir=paths['extracted_data_dir'],
    table_name='PATIENTS.csv',
    usecols=['SUBJECT_ID', 'GENDER', 'DOB'],
    nrows=5,
    low_memory=config['dataset']['low_memory'],
)
patients_preview

## Next notebook

`02_data_exploration.ipynb` will profile table schemas, cohort counts, note coverage, and missingness patterns needed for rigorous cohort construction.